In [3]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# =====================================================
# 1. CONFIGURATION
# =====================================================

DATA_PATH = r"C:\Users\Sam\Desktop\ML\task\Data.xlsx"
sheet_name = "Data_after_KFold_QR"

CONFIG = {
    "optimizer": "BOA",
    "population": 25,
    "iterations": 200,
    "cv": 5,
    "random_state": 42
}

# =====================================================
# 2. LOAD DATA
# =====================================================

df = pd.read_excel(DATA_PATH, sheet_name=sheet_name)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=CONFIG["random_state"]
)

# =====================================================
# 3. MODEL DEFINITION (QR)
# =====================================================

MODEL = {
    "name": "Quantile Regression (QR)",
    "builder": GradientBoostingRegressor,
    "bounds": {
        "n_estimators": (50, 300, int),
        "max_depth": (2, 6, int),
        "learning_rate": (0.01, 0.3, float)
    }
}

# =====================================================
# 4. HELPER FUNCTIONS
# =====================================================

def bounds_to_arrays(bounds):
    lb, ub, cast = [], [], []
    for v in bounds.values():
        lb.append(v[0])
        ub.append(v[1])
        cast.append(v[2])
    return np.array(lb), np.array(ub), cast


def decode_params(vec, bounds, cast):
    decoded = {}
    for i, k in enumerate(bounds.keys()):
        decoded[k] = cast[i](vec[i])
    return decoded


def make_objective(model_builder, bounds, cast):
    def objective(vec):
        params = decode_params(vec, bounds, cast)

        model = model_builder(
            **params,
            loss="quantile",
            alpha=0.5,  # median quantile
            random_state=CONFIG["random_state"]
        )

        neg_mse = cross_val_score(
            model,
            X_train,
            y_train,
            cv=CONFIG["cv"],
            scoring="neg_mean_squared_error",
            n_jobs=-1
        ).mean()

        rmse = np.sqrt(-neg_mse)
        return rmse  # minimize RMSE

    return objective

# =====================================================
# 5. BOA OPTIMIZER (RMSE-BASED)
# =====================================================

def BOA(objective, lb, ub, N, T, cast):
    start = time.time()
    D = len(lb)

    pop = lb + np.random.rand(N, D) * (ub - lb)
    fit = np.array([objective(pop[i]) for i in range(N)])

    best_idx = np.argmin(fit)
    best = pop[best_idx].copy()
    best_fit = fit[best_idx]

    convergence = []
    log = []

    for t in range(T):
        alpha = 1 - t / T
        mean_pop = np.mean(pop, axis=0)

        for i in range(N):
            candidate = pop[i] + alpha * np.random.randn(D) * (mean_pop - pop[i])
            candidate = np.clip(candidate, lb, ub)
            f = objective(candidate)

            if f < fit[i]:
                pop[i] = candidate
                fit[i] = f
                if f < best_fit:
                    best, best_fit = candidate.copy(), f

        convergence.append(best_fit)

        best_decoded = decode_params(best, MODEL["bounds"], cast)
        log.append([t + 1] + [best_decoded[k] for k in MODEL["bounds"]] + [best_fit])

        print(
            f"Iter {t+1:03d} | "
            + ", ".join(f"{k}={v}" for k, v in best_decoded.items())
            + f" | RMSE = {best_fit:.6f}"
        )

    runtime = time.time() - start
    return decode_params(best, MODEL["bounds"], cast), best_fit, convergence, runtime, log

# =====================================================
# 6. RUN OPTIMIZATION
# =====================================================

lb, ub, cast = bounds_to_arrays(MODEL["bounds"])
objective = make_objective(MODEL["builder"], MODEL["bounds"], cast)

best_params, best_rmse, convergence, runtime, log = BOA(
    objective,
    lb,
    ub,
    CONFIG["population"],
    CONFIG["iterations"],
    cast
)

# =====================================================
# 7. FINAL MODEL TRAINING
# =====================================================

final_model = MODEL["builder"](
    **best_params,
    loss="quantile",
    alpha=0.5,
    random_state=CONFIG["random_state"]
)

final_model.fit(X_train, y_train)

y_pred_test = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

# =====================================================
# 8. TABLES
# =====================================================

iter_cols = ["iteration"] + list(MODEL["bounds"].keys()) + ["best_cv_rmse"]
iterations_df = pd.DataFrame(log, columns=iter_cols)

convergence_df = pd.DataFrame({"best_rmse": convergence})

summary_df = pd.DataFrame([{
    "Model": MODEL["name"],
    "Optimizer": CONFIG["optimizer"],
    "Best_CV_RMSE": best_rmse,
    "Test_RMSE": test_rmse,
    "Runtime_sec": runtime
}])

best_params_df = pd.DataFrame({
    "parameters": list(best_params.keys()),
    "values": list(best_params.values())
})

# =====================================================
# 9. PRINT RESULTS
# =====================================================

print("\n✅ Best Hyperparameters:")
print(best_params_df)

print("\n✅ Summary:")
print(summary_df)

print("\n✅ Iteration Log Preview:")
print(iterations_df.head(10))

print("\n✅ Convergence Preview:")
print(convergence_df.head(10))


KeyboardInterrupt: 

In [7]:
iterations_df.to_clipboard(index=False)

In [8]:
convergence_df.to_clipboard(index=False)

In [9]:

summary_df.to_clipboard(index=False)

In [10]:
best_params_df.to_clipboard(index=False)